# LightGBM-Huber — rebalance frequency × signal delay

This standalone notebook keeps the original **9 features, Huber LightGBM, and walk-forward folds**, then evaluates six execution rules. All test rows are scored, so future-return availability never determines which names are eligible.

## Six strategies

We test every combination of `rebalance_days ∈ {1, 5, 10}` and `signal_delay_days ∈ {0, 1}`. Rebalancing is counted in global out-of-sample market dates.

- **Delay 0:** the score formed at close $t$ selects the names earning `target_return(t) = TRET_T1D(t+1)`.
- **Delay 1:** the same close-$t$ score first earns `target_return(t+1) = TRET_T1D(t+2)`.
- **5/10-day rebalance:** the selected top/bottom names stay unchanged until the next activation date. Daily leg returns are equal-weight means over those fixed names.

A leg-day is scored only when at least `min_return_coverage` of its selected names have a realized return. Results are gross: no costs, borrow, or market impact.

## Model

Features are `return_1d`, compounded returns over 5/21/63/126/252 days, 21/63-day volatility, and `turnover_21day`. They are passed to LightGBM in raw units—**no standardization, ranking, or winsorization**.

## 1. Configuration

In [ ]:
from __future__ import annotations
from dataclasses import asdict, dataclass
from pathlib import Path
import json, platform, warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

@dataclass
class Config:
    data_path: Path = Path("manager_holdings/R1000_R2000_daily_turnover21D.parquet")
    output_dir: Path = Path("lightgbm_rv_turnover_execution_results")
    wf_train_months: int = 36
    wf_valid_months: int = 12
    wf_test_months: int = 12
    wf_first_test: str | None = None
    wf_last_test: str | None = None
    wf_expanding: bool = False
    rebalance_days: tuple = (1, 5, 10)
    signal_delay_days: tuple = (0, 1)
    top_k: int = 10
    min_return_coverage: float = 0.8
    hac_lags: int = 5
    n_estimators: int = 500
    early_stopping_rounds: int = 30
    learning_rate: float = 0.05
    num_leaves: int = 63
    huber_alpha: float = 0.9
    subsample: float = 0.8
    colsample_bytree: float = 0.9
    reg_lambda: float = 1.0
    max_bin: int = 127
    seed: int = 1337
    n_jobs: int = -1

    def validate(self):
        if min(self.wf_train_months, self.wf_valid_months, self.wf_test_months) < 1:
            raise ValueError("walk-forward windows must be positive")
        if not self.rebalance_days or min(self.rebalance_days) < 1:
            raise ValueError("rebalance_days must contain positive integers")
        if set(self.signal_delay_days) - {0, 1}:
            raise ValueError("signal_delay_days must contain only 0 and/or 1")
        if self.top_k < 1 or not 0 < self.min_return_coverage <= 1:
            raise ValueError("top_k must be positive and coverage must be in (0, 1]")
        return self

CFG = Config().validate()
FEATURES = [
    "return_1d", "return_5d", "return_21d", "return_63d",
    "return_126d", "return_252d", "volatility_21d",
    "volatility_63d", "turnover_21day",
]
DIAGNOSTIC_COLUMNS = ["DOLLARVOLUME_AVG21D", "market_cap"]
display(pd.Series({
    "data_path": str(CFG.data_path),
    "train / valid / test": f"{CFG.wf_train_months}m / {CFG.wf_valid_months}m / {CFG.wf_test_months}m",
    "strategies": len(CFG.rebalance_days) * len(CFG.signal_delay_days),
    "top_k per leg": CFG.top_k,
}, name="value").to_frame())

## 2. Portfolio engine

`strategy_path` separates the **signal date** from the **activation date**. Security names are selected once per rebalance and carried unchanged through that holding block.

In [ ]:
def strategy_path(score_panel, return_panel, rebalance_days, signal_delay_days,
                  top_k, min_return_coverage=0.0):
    """Build one fixed-name top/bottom path on global OOS market dates."""
    if rebalance_days < 1 or signal_delay_days not in (0, 1):
        raise ValueError("rebalance_days >= 1 and signal_delay_days in {0, 1} required")
    if score_panel.duplicated(["day", "security"]).any():
        raise ValueError("score_panel has duplicated (day, security) rows")
    if return_panel.duplicated(["day", "security"]).any():
        raise ValueError("return_panel has duplicated (day, security) rows")
    days = pd.Index(pd.to_datetime(score_panel["day"].drop_duplicates()).sort_values())
    scores_by_day = {d: g.dropna(subset=["score"])
                     for d, g in score_panel.groupby("day", sort=False)}
    returns_by_day = {d: g.set_index("security")["target_return"]
                      for d, g in return_panel.groupby("day", sort=False)}
    rows, events = [], []
    previous_top = previous_bottom = None
    for signal_i in range(0, len(days), rebalance_days):
        activation_i = signal_i + signal_delay_days
        if activation_i >= len(days):
            continue
        signal_day = days[signal_i]
        ranked = scores_by_day.get(signal_day, pd.DataFrame())
        if ranked.empty:
            continue
        ranked = ranked.sort_values(["score", "security"],
                                    ascending=[False, True], kind="stable")
        k = min(top_k, len(ranked) // 2)
        if k < 1:
            continue
        top_names = ranked.head(k)["security"].astype(str).tolist()
        bottom_names = ranked.tail(k)["security"].astype(str).tolist()
        top_set, bottom_set = set(top_names), set(bottom_names)
        top_turnover = (np.nan if previous_top is None else
                        1.0 - len(top_set & previous_top) / k)
        bottom_turnover = (np.nan if previous_bottom is None else
                           1.0 - len(bottom_set & previous_bottom) / k)
        events.append({
            "signal_day": signal_day, "activation_day": days[activation_i],
            "top_selected": k, "bottom_selected": k,
            "top_turnover": top_turnover, "bottom_turnover": bottom_turnover,
        })
        previous_top, previous_bottom = top_set, bottom_set
        stop_i = min(activation_i + rebalance_days, len(days))
        for held_i in range(activation_i, stop_i):
            day = days[held_i]
            realized = returns_by_day.get(day, pd.Series(dtype=float))
            top_values = pd.to_numeric(realized.reindex(top_names), errors="coerce")
            bottom_values = pd.to_numeric(realized.reindex(bottom_names), errors="coerce")
            top_coverage = float(top_values.notna().mean())
            bottom_coverage = float(bottom_values.notna().mean())
            top_return = (float(top_values.mean())
                          if top_coverage >= min_return_coverage else np.nan)
            bottom_return = (float(bottom_values.mean())
                             if bottom_coverage >= min_return_coverage else np.nan)
            rows.append({
                "day": day, "signal_day": signal_day,
                "top_return": top_return, "bottom_return": bottom_return,
                "spread_return": top_return - bottom_return,
                "top_coverage": top_coverage, "bottom_coverage": bottom_coverage,
                "top_security": "|".join(top_names),
                "bottom_security": "|".join(bottom_names),
            })
    name = f"rebalance_{rebalance_days}d__delay_{signal_delay_days}d"
    path = pd.DataFrame(rows)
    event_table = pd.DataFrame(events)
    for table in (path, event_table):
        table.insert(0, "strategy", name)
        table.insert(1, "rebalance_days", rebalance_days)
        table.insert(2, "signal_delay_days", signal_delay_days)
    return path, event_table

def hac_t(values, lags):
    a = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
    if len(a) < 2:
        return np.nan
    e = a - a.mean()
    lrv = float(e @ e / len(a))
    for lag in range(1, min(lags, len(a) - 1) + 1):
        weight = 1.0 - lag / (lags + 1.0)
        lrv += 2.0 * weight * float(e[lag:] @ e[:-lag] / len(a))
    return float(a.mean() / np.sqrt(lrv / len(a))) if lrv > 0 else np.nan

def max_drawdown(values):
    returns = pd.to_numeric(values, errors="coerce").dropna()
    if returns.empty or (returns < -1).any():
        return np.nan
    wealth = (1 + returns).cumprod().to_numpy()
    peak = np.maximum.accumulate(np.r_[1.0, wealth])[1:]
    return float(np.min(wealth / peak - 1.0))

def performance(values, hac_lags):
    returns = pd.to_numeric(values, errors="coerce").dropna()
    if returns.empty:
        return {"days": 0, "ann_mean": np.nan, "ann_vol": np.nan,
                "sharpe": np.nan, "max_drawdown": np.nan, "hac_t": np.nan}
    ann_mean = float(returns.mean() * 252)
    ann_vol = float(returns.std(ddof=1) * np.sqrt(252)) if len(returns) > 1 else np.nan
    return {"days": len(returns), "ann_mean": ann_mean, "ann_vol": ann_vol,
            "sharpe": ann_mean / ann_vol if ann_vol > 0 else np.nan,
            "max_drawdown": max_drawdown(returns),
            "hac_t": hac_t(returns, hac_lags)}

def summarize_strategy(path, events, hac_lags):
    rows = []
    years = max((path.day.max() - path.day.min()).days / 365.25, 1 / 252)
    specs = [("top", "top_return", "top_coverage", "top_turnover"),
             ("bottom", "bottom_return", "bottom_coverage", "bottom_turnover"),
             ("spread", "spread_return", None, None)]
    for leg, return_col, coverage_col, turnover_col in specs:
        coverage = (path[coverage_col].mean() if coverage_col else
                    path[["top_coverage", "bottom_coverage"]].min(axis=1).mean())
        turnover = (events[turnover_col].dropna() if turnover_col else
                    events[["top_turnover", "bottom_turnover"]].mean(axis=1).dropna())
        rows.append({
            "strategy": path.strategy.iat[0],
            "rebalance_days": int(path.rebalance_days.iat[0]),
            "signal_delay_days": int(path.signal_delay_days.iat[0]),
            "leg": leg, **performance(path[return_col], hac_lags),
            "mean_return_coverage": float(coverage),
            "rebalances": len(events),
            "mean_turnover_per_rebalance": float(turnover.mean()) if len(turnover) else np.nan,
            "annualized_name_turnover": float(turnover.sum() / years),
        })
    return pd.DataFrame(rows)

In [ ]:
# delay=1 uses yesterday's signal; the selected names then stay fixed for 2 days.
test_days = pd.bdate_range("2024-01-02", periods=5)
test_scores = pd.DataFrame({
    "day": np.repeat(test_days, 2),
    "security": ["A", "B"] * len(test_days),
    "score": [2, 1, 1, 2, 1, 2, 2, 1, 1, 2],
})
test_returns = test_scores[["day", "security"]].copy()
test_returns["target_return"] = np.where(test_returns.security.eq("A"), 0.01, -0.01)
test_path, test_events = strategy_path(
    test_scores, test_returns, rebalance_days=2, signal_delay_days=1, top_k=1
)
assert test_path.day.tolist() == test_days[1:].tolist()
assert test_events.signal_day.tolist() == [test_days[0], test_days[2]]
assert test_events.activation_day.tolist() == [test_days[1], test_days[3]]
assert test_path.top_security.tolist() == ["A", "A", "B", "B"]
assert np.allclose(test_path.top_return, [0.01, 0.01, -0.01, -0.01])
print("portfolio timing checks passed")

## 3. Data and features

`TRET_T1D(t)` is the return ending at close $t$. The label is the same field on the next adjacent global market date. Rolling windows never cross a security-level trading gap.

In [ ]:
REQUIRED = ["day", "security", "TRET_T1D", "turnover_21day", *DIAGNOSTIC_COLUMNS]
if not CFG.data_path.exists():
    raise FileNotFoundError(f"not found: {CFG.data_path.resolve()}")
schema = set(pq.ParquetFile(CFG.data_path).schema.names)
missing = sorted(set(REQUIRED) - schema)
if missing:
    raise ValueError(f"parquet is missing required columns: {missing}")
frame = pd.read_parquet(CFG.data_path, columns=REQUIRED).copy()
frame["day"] = pd.to_datetime(frame["day"], errors="coerce").dt.normalize()
if frame["day"].isna().any() or frame["security"].isna().any():
    raise ValueError("day and security must be non-missing")
frame["security"] = frame["security"].astype("string")
for column in ["TRET_T1D", "turnover_21day", *DIAGNOSTIC_COLUMNS]:
    frame[column] = pd.to_numeric(frame[column], errors="coerce")
if frame.duplicated(["day", "security"]).any():
    raise ValueError("duplicated (day, security) rows")
frame = frame.sort_values(["security", "day"]).reset_index(drop=True)
print(f"{len(frame):,} rows | {frame.security.nunique():,} securities | "
      f"{frame.day.nunique():,} market dates | {frame.day.min().date()} to {frame.day.max().date()}")

In [ ]:
market_dates = pd.Index(frame["day"].drop_duplicates().sort_values())
date_number = pd.Series(np.arange(len(market_dates)), index=market_dates)
frame["_dn"] = frame["day"].map(date_number).astype(int)
raw_return = frame["TRET_T1D"].to_numpy(float)
frame["return_1d"] = raw_return
frame["target_return"] = np.nan
for positions in frame.groupby("security", sort=False).indices.values():
    positions = np.asarray(positions, dtype=int)
    dn = frame["_dn"].to_numpy()[positions]
    adjacent = np.diff(dn) == 1
    frame.loc[positions[:-1][adjacent], "target_return"] = raw_return[positions][1:][adjacent]

def rolling_compound(values, dn, window):
    out = np.full(len(values), np.nan)
    if len(values) < window:
        return out
    factors = 1 + values
    valid = np.isfinite(values) & (factors >= 0)
    zeros = valid & (factors == 0)
    logs = np.zeros(len(values))
    positive = valid & (factors > 0)
    logs[positive] = np.log(factors[positive])
    bad_prefix = np.r_[0, np.cumsum(~valid)]
    zero_prefix = np.r_[0, np.cumsum(zeros)]
    log_prefix = np.r_[0.0, np.cumsum(logs)]
    end = np.arange(window - 1, len(values))
    start = end - window + 1
    usable = ((bad_prefix[end + 1] - bad_prefix[start] == 0) &
              (dn[end] - dn[start] == window - 1))
    zero_count = zero_prefix[end + 1] - zero_prefix[start]
    out[end[usable & (zero_count > 0)]] = -1.0
    normal = usable & (zero_count == 0)
    out[end[normal]] = np.expm1((log_prefix[end + 1] - log_prefix[start])[normal])
    return out

def rolling_volatility(values, dn, window):
    out = np.full(len(values), np.nan)
    if len(values) < window:
        return out
    valid = np.isfinite(values)
    safe = np.where(valid, values, 0.0)
    count = np.r_[0, np.cumsum(valid)]
    total = np.r_[0.0, np.cumsum(safe)]
    square = np.r_[0.0, np.cumsum(safe * safe)]
    end = np.arange(window - 1, len(values))
    start = end - window + 1
    usable = ((count[end + 1] - count[start] == window) &
              (dn[end] - dn[start] == window - 1))
    sum_x = total[end + 1] - total[start]
    sum_x2 = square[end + 1] - square[start]
    variance = np.maximum(sum_x2 / window - (sum_x / window) ** 2, 0.0)
    out[end[usable]] = np.sqrt(variance[usable])
    return out

for window in (5, 21, 63, 126, 252):
    frame[f"return_{window}d"] = np.nan
for window in (21, 63):
    frame[f"volatility_{window}d"] = np.nan
for positions in frame.groupby("security", sort=False).indices.values():
    positions = np.asarray(positions, dtype=int)
    values = frame["return_1d"].to_numpy()[positions]
    dn = frame["_dn"].to_numpy()[positions]
    for window in (5, 21, 63, 126, 252):
        frame.loc[positions, f"return_{window}d"] = rolling_compound(values, dn, window)
    for window in (21, 63):
        frame.loc[positions, f"volatility_{window}d"] = rolling_volatility(values, dn, window)
assert np.isclose(frame["TRET_T1D"], frame["return_1d"], equal_nan=True).all()
display(frame[FEATURES + ["target_return"]].notna().mean().rename("coverage").to_frame().round(4))

## 4. Walk-forward LightGBM

Each fold uses 36 months of training, 12 months of validation, and 12 months of test data. Training and validation require a label; the test scoring universe does **not** use next-day label availability as an eligibility filter.

In [ ]:
def make_folds(data, cfg):
    first_day, last_day = data.day.min(), data.day.max()
    month = pd.DateOffset(months=1)
    earliest = first_day + month * (cfg.wf_train_months + cfg.wf_valid_months)
    cursor = max(earliest, pd.Timestamp(cfg.wf_first_test)) if cfg.wf_first_test else earliest
    last_test = pd.Timestamp(cfg.wf_last_test) if cfg.wf_last_test else last_day
    folds, fold_number = [], 0
    while cursor <= last_test:
        test_end = min(cursor + month * cfg.wf_test_months - pd.Timedelta(days=1), last_test)
        valid_end = cursor - pd.Timedelta(days=1)
        valid_start = cursor - month * cfg.wf_valid_months
        train_end = valid_start - pd.Timedelta(days=1)
        train_start = first_day if cfg.wf_expanding else train_end - month * cfg.wf_train_months
        train_start = max(train_start, first_day)
        if train_end > train_start and test_end >= cursor:
            folds.append({
                "fold": fold_number, "train_start": train_start, "train_end": train_end,
                "valid_start": valid_start, "valid_end": valid_end,
                "test_start": cursor, "test_end": test_end,
            })
            fold_number += 1
        cursor += month * cfg.wf_test_months
    if not folds:
        raise ValueError("no walk-forward folds")
    return folds

def split_fold(data, fold):
    labelled = data.target_return.notna()
    train = data.loc[labelled & data.day.between(fold["train_start"], fold["train_end"])]
    valid = data.loc[labelled & data.day.between(fold["valid_start"], fold["valid_end"])]
    test = data.loc[data.day.between(fold["test_start"], fold["test_end"])]
    return train.copy(), valid.copy(), test.copy()

FOLDS = make_folds(frame, CFG)
fold_rows = []
for fold in FOLDS:
    train, valid, test = split_fold(frame, fold)
    fold_rows.append({
        "fold": fold["fold"],
        "train": f"{fold['train_start'].date()}..{fold['train_end'].date()}",
        "valid": f"{fold['valid_start'].date()}..{fold['valid_end'].date()}",
        "test": f"{fold['test_start'].date()}..{fold['test_end'].date()}",
        "train_rows": len(train), "valid_rows": len(valid), "test_rows": len(test),
        "test_label_coverage": test.target_return.notna().mean(),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table)

In [ ]:
def fit_one(train, valid):
    model = lgb.LGBMRegressor(
        objective="huber", alpha=CFG.huber_alpha, n_estimators=CFG.n_estimators,
        learning_rate=CFG.learning_rate, num_leaves=CFG.num_leaves,
        min_child_samples=min(200, max(1, len(train) // 100)),
        subsample=CFG.subsample, subsample_freq=1,
        colsample_bytree=CFG.colsample_bytree, reg_lambda=CFG.reg_lambda,
        max_bin=CFG.max_bin, random_state=CFG.seed, n_jobs=CFG.n_jobs,
        verbosity=-1, deterministic=True, force_col_wise=True,
    )
    model.fit(
        train[FEATURES].astype("float32"), train.target_return.astype("float32"),
        eval_set=[(valid[FEATURES].astype("float32"), valid.target_return.astype("float32"))],
        eval_metric="l2",
        callbacks=[lgb.early_stopping(CFG.early_stopping_rounds, verbose=False)],
    )
    return model

scored_parts, importance_parts, fit_rows = [], [], []
for fold in FOLDS:
    train, valid, test = split_fold(frame, fold)
    if len(train) < 100 or valid.empty or test.empty:
        warnings.warn(f"fold {fold['fold']} is too thin and was skipped")
        continue
    model = fit_one(train, valid)
    best_iteration = int(model.best_iteration_ or CFG.n_estimators)
    scored_fold = test[["day", "security", "target_return",
                        *FEATURES, *DIAGNOSTIC_COLUMNS]].copy()
    scored_fold["score"] = model.predict(
        test[FEATURES].astype("float32"), num_iteration=best_iteration
    )
    scored_fold["fold"] = fold["fold"]
    scored_parts.append(scored_fold)
    gain = model.booster_.feature_importance(importance_type="gain")
    importance_parts.append(pd.DataFrame({
        "fold": fold["fold"], "feature": FEATURES, "gain": gain
    }))
    fit_rows.append({"fold": fold["fold"], "best_iteration": best_iteration,
                     "train_rows": len(train), "test_days": test.day.nunique()})
    print(f"fold {fold['fold']}: best_iter {best_iteration:>4} | "
          f"train {len(train):>8,} | test {test.day.nunique():>4} days")
if not scored_parts:
    raise RuntimeError("no fold produced predictions")
scored = pd.concat(scored_parts, ignore_index=True).sort_values(["day", "security"])
if scored.duplicated(["day", "security"]).any():
    raise ValueError("test folds overlap: duplicated (day, security) scores")
fit_table = pd.DataFrame(fit_rows)
importance = (pd.concat(importance_parts).groupby("feature", as_index=False)["gain"].mean()
              .assign(gain_share=lambda data: data.gain / data.gain.sum())
              .sort_values("gain", ascending=False).reset_index(drop=True))
display(fit_table)
display(importance.round(6))

## 5. Run the six execution rules

The 1-day rebalance, no-delay rule is the daily execution benchmark. Compare delay within a rebalance frequency, then compare frequencies within a delay column.

In [ ]:
oos_days = pd.Index(scored.day.drop_duplicates().sort_values())
realized_oos = frame.loc[frame.day.isin(oos_days), ["day", "security", "target_return"]]
path_parts, event_parts, summary_parts = [], [], []
for rebalance_days in CFG.rebalance_days:
    for delay_days in CFG.signal_delay_days:
        path, events = strategy_path(
            scored[["day", "security", "score"]], realized_oos,
            rebalance_days=rebalance_days, signal_delay_days=delay_days,
            top_k=CFG.top_k, min_return_coverage=CFG.min_return_coverage,
        )
        path_parts.append(path)
        event_parts.append(events)
        summary_parts.append(summarize_strategy(path, events, CFG.hac_lags))
daily_strategies = pd.concat(path_parts, ignore_index=True)
rebalance_events = pd.concat(event_parts, ignore_index=True)
summary = pd.concat(summary_parts, ignore_index=True)
headline = summary.loc[summary.leg.eq("spread"), [
    "rebalance_days", "signal_delay_days", "days", "ann_mean", "ann_vol",
    "sharpe", "hac_t", "max_drawdown", "mean_return_coverage",
    "mean_turnover_per_rebalance", "annualized_name_turnover",
]].sort_values(["signal_delay_days", "rebalance_days"])
display(headline.round(4))
display(rebalance_events.groupby(["rebalance_days", "signal_delay_days"]).head(1)[[
    "rebalance_days", "signal_delay_days", "signal_day", "activation_day"
]].sort_values(["signal_delay_days", "rebalance_days"]))

year_rows = []
for (strategy, year), values in daily_strategies.groupby(
        ["strategy", daily_strategies.day.dt.year], sort=True):
    year_rows.append({"strategy": strategy, "year": int(year),
                      **performance(values.spread_return, CFG.hac_lags)})
year_table = pd.DataFrame(year_rows)
display(year_table.pivot(index="year", columns="strategy", values="ann_mean").round(4))

In [ ]:
colors = {1: "#355C7D", 5: "#C06C84", 10: "#6C757D"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for axis, delay_days in zip(axes, CFG.signal_delay_days):
    for rebalance_days in CFG.rebalance_days:
        strategy = f"rebalance_{rebalance_days}d__delay_{delay_days}d"
        values = daily_strategies.loc[
            daily_strategies.strategy.eq(strategy), ["day", "spread_return"]
        ].dropna().sort_values("day")
        cumulative = values.spread_return.cumsum()
        axis.plot(values.day, 100 * cumulative, label=f"every {rebalance_days}d",
                  color=colors[rebalance_days], linewidth=1.8)
    axis.axhline(0, color="#222222", linewidth=0.8)
    axis.set_title(f"Signal delay: {delay_days} market day")
    axis.set_xlabel("test date")
    axis.grid(alpha=0.22)
    axis.legend(frameon=False)
axes[0].set_ylabel("cumulative arithmetic spread return (pp)")
fig.suptitle("Cumulative arithmetic top-minus-bottom return", y=1.02)
fig.tight_layout()
plt.show()

## 6. Reading the comparison

1. Compare `delay=0` with `delay=1` at the same rebalance frequency: this isolates one-day signal decay.
2. Compare 1/5/10 days at the same delay: this measures how quickly the ranking becomes stale.
3. Read Sharpe together with HAC t-stat, drawdown, return coverage, and annualized name turnover.
4. A better gross result at lower frequency is not automatically implementable alpha; costs and weight drift are not modeled here.
5. `turnover_21day` is assumed to be a trailing measure known at close $t$; verify its upstream construction separately.

## 7. Save

In [ ]:
CFG.output_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(CFG.output_dir / "strategy_summary.csv", index=False)
year_table.to_csv(CFG.output_dir / "strategy_by_year.csv", index=False)
daily_strategies.to_parquet(CFG.output_dir / "daily_strategies.parquet", index=False)
rebalance_events.to_parquet(CFG.output_dir / "rebalance_events.parquet", index=False)
importance.to_csv(CFG.output_dir / "feature_importance.csv", index=False)
fold_table.to_csv(CFG.output_dir / "folds.csv", index=False)
fit_table.to_csv(CFG.output_dir / "fit_table.csv", index=False)
fig.savefig(CFG.output_dir / "cumulative_spread.png", dpi=180, bbox_inches="tight")
run_config = {
    **{key: str(value) for key, value in asdict(CFG).items()},
    "features": FEATURES, "folds": len(FOLDS),
    "test_dates": int(scored.day.nunique()),
    "python": platform.python_version(), "lightgbm": lgb.__version__,
    "pandas": pd.__version__, "numpy": np.__version__,
}
(CFG.output_dir / "run_config.json").write_text(json.dumps(run_config, indent=2))
print("saved to", CFG.output_dir.resolve())